# Resume Matching and Embedding-Based Ranking

### Objective

This notebook extends the Resume Classifier project with a semantic
resume-to-job matching system.

The notebook will:

- Load the extracted resume dataset.
- Apply the same NLP preprocessing used by the classifier.
- Convert resumes into dense semantic embeddings using Sentence Transformers.
- Define job descriptions representing different roles.
- Convert job descriptions into embeddings using the same model.
- Calculate cosine similarity between each resume and a selected job description.
- Generate a relevance score for every resume.
- Rank candidates based on semantic similarity.
- Analyze the quality and distribution of the matching scores.

### Important distinction

The original classifier uses TF-IDF + XGBoost for job-role classification.

This notebook uses embeddings + cosine similarity for semantic
resume-to-job matching and candidate ranking.

# Resume Matching and Relevance Ranking

This notebook adds a separate relevance-ranking component to the
Resume Classifier project.

It uses two approaches:

1. Embedding-based semantic relevance using Sentence Transformers.
2. Keyword-based relevance using role-specific keywords.

The original XGBoost classifier is not modified by this notebook.

### Pipeline

Resume
   ↓
Sentence Transformer
   ↓
Resume Embedding
   ↓
Cosine Similarity with Role Profile
   ↓
Semantic Relevance Score
   ↓
Ranking

Resume
   ↓
Keyword Matching
   ↓
Matched Role Keywords
   ↓
Keyword Relevance Score
   ↓
Ranking

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

C:\Users\Shaurya Mittal\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "DATA"
RESUME_FILE = DATA_DIR / "Resume.csv"

print("Resume file:", RESUME_FILE)

Resume file: C:\Users\Shaurya Mittal\Documents\ML PROJECTS\Project 1\RESUME CLASSIFIER\DATA\Resume.csv


In [4]:
df = pd.read_csv(RESUME_FILE)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (2484, 4)
Columns: ['ID', 'Resume_str', 'Resume_html', 'Category']


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [6]:
print("Number of categories:", df["Category"].nunique())

print("\nCategories:")
for category in sorted(df["Category"].unique()):
    print(category)

Number of categories: 24

Categories:
ACCOUNTANT
ADVOCATE
AGRICULTURE
APPAREL
ARTS
AUTOMOBILE
AVIATION
BANKING
BPO
BUSINESS-DEVELOPMENT
CHEF
CONSTRUCTION
CONSULTANT
DESIGNER
DIGITAL-MEDIA
ENGINEERING
FINANCE
FITNESS
HEALTHCARE
HR
INFORMATION-TECHNOLOGY
PUBLIC-RELATIONS
SALES
TEACHER


In [7]:
def prepare_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["embedding_text"] = df["Resume_str"].apply(prepare_text)

df = df[df["embedding_text"].str.len() > 0].copy()

print("Usable resumes:", len(df))

Usable resumes: 2483


In [8]:
ROLE_KEYWORDS = {
    "HR": [
        "recruitment",
        "employee relations",
        "payroll",
        "benefits",
        "talent acquisition",
        "training",
        "compliance",
    ],

    "DESIGNER": [
        "graphic design",
        "photoshop",
        "illustrator",
        "creativity",
        "branding",
        "layout",
        "typography",
    ],

    "INFORMATION-TECHNOLOGY": [
        "software development",
        "programming",
        "databases",
        "networks",
        "troubleshooting",
        "cybersecurity",
    ],

    "TEACHER": [
        "lesson planning",
        "classroom management",
        "curriculum development",
        "student assessment",
        "pedagogy",
    ],

    "ADVOCATE": [
        "legal advice",
        "litigation",
        "contracts",
        "court representation",
        "legal research",
        "case management",
    ],

    "BUSINESS-DEVELOPMENT": [
        "sales strategy",
        "partnerships",
        "market expansion",
        "lead generation",
        "client acquisition",
    ],

    "HEALTHCARE": [
        "patient care",
        "diagnosis",
        "treatment",
        "medical terminology",
        "clinical support",
        "healthcare services",
    ],

    "FITNESS": [
        "personal training",
        "workout plans",
        "nutrition",
        "physical health",
        "exercise coaching",
        "strength",
    ],

    "AGRICULTURE": [
        "farming",
        "crop production",
        "irrigation",
        "soil management",
        "pest control",
        "harvesting",
    ],

    "BPO": [
        "call center",
        "customer service",
        "communication",
        "data entry",
        "process outsourcing",
        "support",
    ],

    "SALES": [
        "sales targets",
        "customer relationship",
        "CRM",
        "lead conversion",
        "cold calling",
        "negotiation",
    ],

    "CONSULTANT": [
        "strategy",
        "analysis",
        "business solutions",
        "client management",
        "project execution",
        "problem solving",
    ],

    "DIGITAL-MEDIA": [
        "social media",
        "marketing",
        "content creation",
        "SEO",
        "analytics",
        "branding",
        "engagement",
    ],

    "AUTOMOBILE": [
        "vehicle maintenance",
        "automotive engineering",
        "repair",
        "diagnostics",
        "manufacturing",
        "mechanics",
    ],

    "CHEF": [
        "cooking",
        "recipes",
        "food preparation",
        "kitchen management",
        "menu planning",
        "ingredients",
    ],

    "FINANCE": [
        "financial analysis",
        "budgeting",
        "accounting",
        "investment",
        "risk management",
        "auditing",
    ],

    "APPAREL": [
        "fashion",
        "clothing",
        "textiles",
        "garment design",
        "merchandising",
        "production",
        "trends",
    ],

    "ENGINEERING": [
        "mechanical",
        "electrical",
        "civil",
        "CAD",
        "systems design",
        "problem solving",
        "development",
    ],

    "ACCOUNTANT": [
        "balance sheet",
        "taxation",
        "auditing",
        "bookkeeping",
        "financial reporting",
        "ledger entries",
    ],

    "CONSTRUCTION": [
        "site management",
        "blueprints",
        "structural engineering",
        "safety",
        "equipment",
        "planning",
        "labor",
    ],

    "PUBLIC-RELATIONS": [
        "media communication",
        "brand image",
        "press releases",
        "crisis management",
        "campaigns",
    ],

    "BANKING": [
        "loans",
        "credit analysis",
        "customer service",
        "transactions",
        "investment banking",
        "regulations",
    ],

    "ARTS": [
        "painting",
        "sculpture",
        "performing arts",
        "creativity",
        "exhibitions",
        "visual storytelling",
    ],

    "AVIATION": [
        "flight operations",
        "aircraft maintenance",
        "aviation safety",
        "airport",
        "customer service",
    ],
}

In [9]:
role_profiles = {
    role: " ".join(keywords)
    for role, keywords in ROLE_KEYWORDS.items()
}

role_df = pd.DataFrame(
    {
        "Category": list(role_profiles.keys()),
        "Role_Profile": list(role_profiles.values()),
    }
)

role_df.head()

,Category,Role_Profile
0,HR,recruitment employee relations payroll benefit...
1,DESIGNER,graphic design photoshop illustrator creativit...
2,INFORMATION-TECHNOLOGY,software development programming databases net...
3,TEACHER,lesson planning classroom management curriculu...
4,ADVOCATE,legal advice litigation contracts court repres...


In [10]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Loaded:", EMBEDDING_MODEL)

Loaded: all-MiniLM-L6-v2


In [11]:
resume_embeddings = embedding_model.encode(
    df["embedding_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Resume embedding shape:")
print(resume_embeddings.shape)

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 78/78 [02:56<00:00,  2.27s/it]

Resume embedding shape:
(2483, 384)


In [12]:
role_embeddings = embedding_model.encode(
    role_df["Role_Profile"].tolist(),
    normalize_embeddings=True
)

print("Role embedding shape:")
print(role_embeddings.shape)

Role embedding shape:
(24, 384)


In [13]:
semantic_scores = cosine_similarity(
    resume_embeddings,
    role_embeddings
)

print("Similarity matrix shape:")
print(semantic_scores.shape)

Similarity matrix shape:
(2483, 24)


In [14]:
best_role_indices = semantic_scores.argmax(axis=1)

df["Embedding_Role"] = [
    role_df.iloc[i]["Category"]
    for i in best_role_indices
]

df["Embedding_Relevance_Score"] = semantic_scores.max(
    axis=1
)

df[
    [
        "ID",
        "Category",
        "Embedding_Role",
        "Embedding_Relevance_Score"
    ]
].head(10)

,ID,Category,Embedding_Role,Embedding_Relevance_Score
0,16852973,HR,HR,0.459136
1,22323967,HR,HR,0.441831
2,33176873,HR,HR,0.466657
3,27018550,HR,BPO,0.439063
4,17812897,HR,HR,0.496840
5,11592605,HR,HR,0.364514
6,25824789,HR,HR,0.480047
7,15375009,HR,HR,0.354790
8,11847784,HR,HR,0.504816
9,32896934,HR,HR,0.497963


In [15]:
embedding_ranking = []

for role_index, role in enumerate(role_df["Category"]):

    scores = semantic_scores[:, role_index]

    temp = df[
        ["ID", "Category"]
    ].copy()

    temp["Matched_Role"] = role
    temp["Embedding_Relevance_Score"] = scores

    temp = temp.sort_values(
        "Embedding_Relevance_Score",
        ascending=False
    ).reset_index(drop=True)

    temp["Embedding_Rank"] = np.arange(
        1,
        len(temp) + 1
    )

    embedding_ranking.append(temp)

embedding_ranking_df = pd.concat(
    embedding_ranking,
    ignore_index=True
)

embedding_ranking_df.head()

,ID,Category,Matched_Role,Embedding_Relevance_Score,Embedding_Rank
0,30862904,HR,HR,0.547460,1
1,26780935,HR,HR,0.544035,2
2,73077810,HR,HR,0.543490,3
3,24184357,HR,HR,0.540161,4
4,87968870,HR,HR,0.536713,5


In [16]:
top_it = (
    embedding_ranking_df[
        embedding_ranking_df["Matched_Role"]
        == "INFORMATION-TECHNOLOGY"
    ]
    .head(10)
)

top_it

,ID,Category,Matched_Role,Embedding_Relevance_Score,Embedding_Rank
4966,18187364,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.509637,1
4967,12144825,AVIATION,INFORMATION-TECHNOLOGY,0.475667,2
4968,20237244,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.471442,3
4969,16186411,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.459744,4
4970,12374933,CONSULTANT,INFORMATION-TECHNOLOGY,0.449728,5
4971,31395742,CONSULTANT,INFORMATION-TECHNOLOGY,0.439571,6
4972,24643412,CONSULTANT,INFORMATION-TECHNOLOGY,0.429139,7
4973,43378989,CONSULTANT,INFORMATION-TECHNOLOGY,0.426534,8
4974,31813535,BUSINESS-DEVELOPMENT,INFORMATION-TECHNOLOGY,0.414606,9
4975,26746496,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.409438,10


In [17]:
def normalize_for_keywords(text):
    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def calculate_keyword_score(
    resume_text,
    keywords
):
    resume_text = normalize_for_keywords(
        resume_text
    )

    matched_keywords = []

    for keyword in keywords:

        keyword = normalize_for_keywords(
            keyword
        )

        pattern = (
            r"\b"
            + re.escape(keyword)
            + r"\b"
        )

        if re.search(pattern, resume_text):
            matched_keywords.append(keyword)

    total_keywords = len(keywords)

    if total_keywords == 0:
        return 0.0, []

    score = (
        len(matched_keywords)
        / total_keywords
    )

    return score, matched_keywords

In [18]:
def normalize_for_keywords(text):
    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def calculate_keyword_score(
    resume_text,
    keywords
):
    resume_text = normalize_for_keywords(
        resume_text
    )

    matched_keywords = []

    for keyword in keywords:

        keyword = normalize_for_keywords(
            keyword
        )

        pattern = (
            r"\b"
            + re.escape(keyword)
            + r"\b"
        )

        if re.search(pattern, resume_text):
            matched_keywords.append(keyword)

    total_keywords = len(keywords)

    if total_keywords == 0:
        return 0.0, []

    score = (
        len(matched_keywords)
        / total_keywords
    )

    return score, matched_keywords

In [20]:
keyword_ranking = []

for role, keywords in ROLE_KEYWORDS.items():

    for _, row in df.iterrows():

        score, matched = calculate_keyword_score(
            row["Resume_str"],
            keywords
        )

        keyword_ranking.append(
            {
                "ID": row["ID"],
                "Actual_Category": row["Category"],
                "Matched_Role": role,
                "Keyword_Relevance_Score": score,
                "Matched_Keywords": ", ".join(matched),
            }
        )

keyword_ranking_df = pd.DataFrame(
    keyword_ranking
)

keyword_ranking_df.head()

,ID,Actual_Category,Matched_Role,Keyword_Relevance_Score,Matched_Keywords
0,16852973,HR,HR,0.571429,"employee relations, payroll, benefits, training"
1,22323967,HR,HR,0.428571,"recruitment, benefits, training"
2,33176873,HR,HR,0.714286,"recruitment, payroll, benefits, training, comp..."
3,27018550,HR,HR,0.285714,"payroll, training"
4,17812897,HR,HR,0.857143,"recruitment, employee relations, payroll, bene..."


In [21]:
keyword_ranking_df = (
    keyword_ranking_df
    .sort_values(
        [
            "Matched_Role",
            "Keyword_Relevance_Score"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

keyword_ranking_df["Keyword_Rank"] = (
    keyword_ranking_df
    .groupby("Matched_Role")
    .cumcount()
    + 1
)

keyword_ranking_df.head(20)

,ID,Actual_Category,Matched_Role,Keyword_Relevance_Score,Matched_Keywords,Keyword_Rank
0,12065211,ACCOUNTANT,ACCOUNTANT,0.833333,"balance sheet, taxation, auditing, bookkeeping...",1
1,81677620,FINANCE,ACCOUNTANT,0.666667,"balance sheet, taxation, auditing, ledger entries",2
2,24670867,FINANCE,ACCOUNTANT,0.666667,"balance sheet, taxation, auditing, financial r...",3
3,37370455,ACCOUNTANT,ACCOUNTANT,0.666667,"balance sheet, taxation, bookkeeping, financia...",4
4,23636277,ACCOUNTANT,ACCOUNTANT,0.666667,"balance sheet, auditing, bookkeeping, financia...",5
5,59403481,ACCOUNTANT,ACCOUNTANT,0.666667,"balance sheet, auditing, bookkeeping, financia...",6
6,23387174,ACCOUNTANT,ACCOUNTANT,0.666667,"balance sheet, taxation, auditing, financial r...",7
7,14287992,BUSINESS-DEVELOPMENT,ACCOUNTANT,0.500000,"balance sheet, financial reporting, ledger ent...",8
8,21568833,CONSULTANT,ACCOUNTANT,0.500000,"balance sheet, taxation, auditing",9
9,95792386,CONSULTANT,ACCOUNTANT,0.500000,"taxation, auditing, financial reporting",10


In [22]:
top_keyword_it = (
    keyword_ranking_df[
        keyword_ranking_df["Matched_Role"]
        == "INFORMATION-TECHNOLOGY"
    ]
    .head(10)
)

top_keyword_it

,ID,Actual_Category,Matched_Role,Keyword_Relevance_Score,Matched_Keywords,Keyword_Rank
49660,90867631,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.833333,"software development, programming, databases, ...",1
49661,12635195,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.666667,"software development, programming, networks, t...",2
49662,51639418,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.666667,"software development, programming, databases, ...",3
49663,20879311,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.666667,"databases, networks, troubleshooting, cybersec...",4
49664,27058381,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.666667,"programming, databases, networks, troubleshooting",5
49665,39413067,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.666667,"software development, databases, networks, tro...",6
49666,11020140,CONSULTANT,INFORMATION-TECHNOLOGY,0.666667,"software development, programming, databases, ...",7
49667,24643412,CONSULTANT,INFORMATION-TECHNOLOGY,0.666667,"software development, programming, databases, ...",8
49668,12374933,CONSULTANT,INFORMATION-TECHNOLOGY,0.666667,"programming, databases, networks, troubleshooting",9
49669,65062795,CONSULTANT,INFORMATION-TECHNOLOGY,0.666667,"software development, programming, databases, ...",10


In [23]:
combined_df = embedding_ranking_df.merge(
    keyword_ranking_df[
        [
            "ID",
            "Matched_Role",
            "Keyword_Relevance_Score",
            "Matched_Keywords"
        ]
    ],
    on=["ID", "Matched_Role"],
    how="inner"
)

In [24]:
combined_df = embedding_ranking_df.merge(
    keyword_ranking_df[
        [
            "ID",
            "Matched_Role",
            "Keyword_Relevance_Score",
            "Matched_Keywords"
        ]
    ],
    on=["ID", "Matched_Role"],
    how="inner"
)

In [26]:
combined_df["Embedding_Normalized"] = (
    combined_df
    .groupby("Matched_Role")[
        "Embedding_Relevance_Score"
    ]
    .transform(
        lambda x: (
            x - x.min()
        ) / (
            x.max() - x.min()
            if x.max() != x.min()
            else 1
        )
    )
)

In [27]:
combined_df["Combined_Relevance_Score"] = (
    0.70 * combined_df["Embedding_Normalized"]
    +
    0.30 * combined_df["Keyword_Relevance_Score"]
)

In [28]:
combined_df = (
    combined_df
    .sort_values(
        [
            "Matched_Role",
            "Combined_Relevance_Score"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

combined_df["Final_Rank"] = (
    combined_df
    .groupby("Matched_Role")
    .cumcount()
    + 1
)

combined_df.head(20)

,ID,Category,Matched_Role,Embedding_Relevance_Score,Embedding_Rank,Keyword_Relevance_Score,Matched_Keywords,Embedding_Normalized,Combined_Relevance_Score,Final_Rank
0,12065211,ACCOUNTANT,ACCOUNTANT,0.658351,1,0.833333,"balance sheet, taxation, auditing, bookkeeping...",1.000000,0.950000,1
1,23636277,ACCOUNTANT,ACCOUNTANT,0.571627,15,0.666667,"balance sheet, auditing, bookkeeping, financia...",0.893087,0.825161,2
2,37370455,ACCOUNTANT,ACCOUNTANT,0.550666,23,0.666667,"balance sheet, taxation, bookkeeping, financia...",0.867246,0.807072,3
3,23387174,ACCOUNTANT,ACCOUNTANT,0.536788,32,0.666667,"balance sheet, taxation, auditing, financial r...",0.850137,0.795096,4
4,14491649,ACCOUNTANT,ACCOUNTANT,0.628364,2,0.333333,"balance sheet, financial reporting",0.963032,0.774122,5
5,20624984,ACCOUNTANT,ACCOUNTANT,0.620512,4,0.333333,"balance sheet, auditing",0.953352,0.767346,6
6,21338490,ACCOUNTANT,ACCOUNTANT,0.560127,19,0.500000,"balance sheet, bookkeeping, ledger entries",0.878909,0.765236,7
7,23513618,ACCOUNTANT,ACCOUNTANT,0.611953,5,0.333333,"auditing, financial reporting",0.942800,0.759960,8
8,25462793,ACCOUNTANT,ACCOUNTANT,0.603460,6,0.333333,"taxation, bookkeeping",0.932331,0.752632,9
9,14470533,ACCOUNTANT,ACCOUNTANT,0.588783,8,0.333333,"balance sheet, bookkeeping",0.914236,0.739965,10


In [30]:
embedding_output = DATA_DIR / "embedding_resume_ranking.csv"
keyword_output = DATA_DIR / "keyword_resume_ranking.csv"
combined_output = DATA_DIR / "combined_resume_ranking.csv"

embedding_ranking_df.to_csv(
    embedding_output,
    index=False
)

keyword_ranking_df.to_csv(
    keyword_output,
    index=False
)

combined_df.to_csv(
    combined_output,
    index=False
)

print("Saved:")
print(embedding_output)
print(keyword_output)
print(combined_output)

Saved:
C:\Users\Shaurya Mittal\Documents\ML PROJECTS\Project 1\RESUME CLASSIFIER\DATA\embedding_resume_ranking.csv
C:\Users\Shaurya Mittal\Documents\ML PROJECTS\Project 1\RESUME CLASSIFIER\DATA\keyword_resume_ranking.csv
C:\Users\Shaurya Mittal\Documents\ML PROJECTS\Project 1\RESUME CLASSIFIER\DATA\combined_resume_ranking.csv
